In [1]:
import streamlit as st
import pandas as pd
import mysql.connector
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Streamlit UI
st.title("Bus Route Scraper & MySQL Uploader")

uploaded_file = st.file_uploader("All_routes_buses.csv", type=["csv"])

if uploaded_file:
    df = pd.read_csv("/home/vasan/Downloads/Python/Project_Red_Bus/All_routes_buses.csv")
    st.success("CSV uploaded successfully.")

    if st.button("Scrape and Insert into MySQL"):
        driver = webdriver.Chrome()
        driver.maximize_window()
        all_buses_data = []

    for i,r in df.iterrows():
        link=r["route_link"]
        route=r["route_name"]

        # Loop through each link
        driver.get(link)
        time.sleep(3)
        
        
        try:
            button = driver.find_elements(By.XPATH, "//div[@class='button']")
            button[0].click()
            time.sleep(3)
        except Exception as e:
            print("No button found or unable to click:", e)
            

        bus_names = driver.find_elements(By.XPATH, "//div[@class='travels lh-24 f-bold d-color']")
        bus_types = driver.find_elements(By.XPATH, "//div[@class='bus-type f-12 m-top-16 l-color evBus']")
        departing_times = driver.find_elements(By.XPATH, "//div[@class='dp-time f-19 d-color f-bold']")
        duration_times = driver.find_elements(By.XPATH, "//div[@class='dur l-color lh-24']")
        reaching_times = driver.find_elements(By.XPATH, "//div[@class='bp-time f-19 d-color disp-Inline']")
        star_ratings = driver.find_elements(By.XPATH, "//div[@class='lh-18 rating rat-green ']")
        prices = driver.find_elements(By.XPATH, "//div[@class='fare d-block']")
        seats_availabilities = driver.find_elements(By.XPATH, "//div[@class='seat-left m-top-16']")
        #print(bus_name,bus_type,departing_time,duration_time,reaching_time,star_rating,price,seats_availability)


        for idx in range(len(bus_names)):
            bus_data = {
                "route": route,
                "link": link,
                "Bus Name": bus_names[idx].text,
                "Bus Type": bus_types[idx].text if idx < len(bus_types) else "N/A",
                "Departing Time": departing_times[idx].text if idx < len(departing_times) else "N/A",
                "Duration Time": duration_times[idx].text if idx < len(duration_times) else "N/A",
                "Reaching Time": reaching_times[idx].text if idx < len(reaching_times) else "N/A",
                "Star Rating": star_ratings[idx].text if idx < len(star_ratings) else "N/A",
                "Price": prices[idx].text if idx < len(prices) else "N/A",
                "Seats Availability": seats_availabilities[idx].text if idx < len(seats_availabilities) else "N/A",
            }
            all_buses_data.append(bus_data)
    driver.quit()
    data_df = pd.DataFrame(all_buses_data)
    st.success("Scraping completed successfully.")
    st.dataframe(data_df)

        # Data filters
    st.write("### Filter Data")
    selected_route = st.selectbox("Select Route", ["All"] + list(data_df["Route"].unique()))
    selected_type = st.selectbox("Select Bus Type", ["All"] + list(data_df["Bus Type"].unique()))

    filtered_df = data_df.copy()
    if selected_route != "All":
        filtered_df = filtered_df[filtered_df["Route"] == selected_route]
    if selected_type != "All":
        filtered_df = filtered_df[filtered_df["Bus Type"] == selected_type]

    st.write("### Filtered Data", filtered_df)

    # Analysis
    def extract_price(p):
        try:
            return float(p.replace("INR", "").replace("₹", "").replace(",", "").strip())
        except:
            return None

    filtered_df["Price Num"] = filtered_df["Price"].apply(extract_price)
    filtered_df["Rating Num"] = pd.to_numeric(filtered_df["Star Rating"], errors="coerce")

    st.write("### Summary")
    st.metric("Average Price", f"₹{filtered_df['Price Num'].mean():.2f}" if not filtered_df['Price Num'].isnull().all() else "N/A")
    st.metric("Average Rating", f"{filtered_df['Rating Num'].mean():.2f}" if not filtered_df['Rating Num'].isnull().all() else "N/A")

    # Insert to MySQL
    if st.button("Upload to MySQL"):
        try:
            conn = mysql.connector.connect(
                host='localhost',
                user='root',
                password='Dheeraj@123',
                database='vasan',
                port=3306
            )
            cursor = conn.cursor()

            create_query = """
            CREATE TABLE IF NOT EXISTS bus_routes (
                id INT AUTO_INCREMENT PRIMARY KEY,
                route_name TEXT,
                route_link TEXT,
                busname TEXT,
                bustype TEXT,
                price DECIMAL(10,2),
                seats_available INT,
                departing_time TIME,
                duration TEXT,
                reaching_time TIME,
                star_rating FLOAT
            );
            """
            cursor.execute(create_query)

            insert_query = """
            INSERT INTO bus_routes (
                route_name, route_link, busname, bustype, price, seats_available,
                departing_time, duration, reaching_time, star_rating
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """

            for _, row in filtered_df.iterrows():
                price = extract_price(row["Price"])
                seats = int(row["Seats Available"].split()[0]) if "Seats" in row["Seats Available"] else None
                star = float(row["Star Rating"]) if row["Star Rating"] != "N/A" else None

                values = (
                    row["Route"], row["Link"], row["Bus Name"], row["Bus Type"],
                    price, seats, row["Departing Time"], row["Duration"],
                    row["Reaching Time"], star
                )
                cursor.execute(insert_query, values)

            conn.commit()
            st.success("✅ Data uploaded to MySQL successfully.")

        except Exception as e:
            st.error(f"❌ MySQL Upload Failed: {e}")
    
    print(r)

2025-05-01 19:27:36.219 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 19:27:36.330 
  command:

    streamlit run /home/vasan/Downloads/Python/Project_Red_Bus/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-05-01 19:27:36.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 19:27:36.331 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 19:27:36.331 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 19:27:36.331 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 19:27:36.332 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01

In [2]:
print(r)

NameError: name 'r' is not defined

In [3]:
if "route_link" in r:
    link = r["route_link"]
else:
    link = None  # or handle accordingly


NameError: name 'r' is not defined

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("/home/vasan/Downloads/Python/Project_Red_Bus/All_routes_buses.csv")
df

,route,link,Bus Name,Bus Type,Departing Time,Duration Time,Reaching Time,Star Rating,Price,Seats Availability
0,Pune to Goa,https://www.redbus.in/bus-tickets/pune-to-goa,Zingbus Plus,Bharat Benz A/C Sleeper (2+1),22:30,11h 10m,09:40,4.5,3077,NaN
1,Pune to Goa,https://www.redbus.in/bus-tickets/pune-to-goa,Zingbus Plus,Bharat Benz A/C Sleeper (2+1),21:45,11h 10m,08:55,4.3,2954,NaN
2,Pune to Goa,https://www.redbus.in/bus-tickets/pune-to-goa,Ashray Grand,Volvo Multi-Axle A/C Sleeper (2+1),21:30,12h 30m,10:00,4.1,INR 3400,NaN
3,Pune to Goa,https://www.redbus.in/bus-tickets/pune-to-goa,AdAnand Tours And Travels,Volvo Multi-Axle I-Shift A/C Sleeper (2+1),20:00,12h 45m,08:45,4.3,INR 3200,NaN
4,Pune to Goa,https://www.redbus.in/bus-tickets/pune-to-goa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
411,Goa to Kolhapur(Maharashtra),https://www.redbus.in/bus-tickets/goa-to-kolha...,Kelkar Travels Garuda,Non A/C Seater / Sleeper (2+1),19:50,06h 40m,02:30,NaN,INR 600,NaN
412,Solapur to Goa,https://www.redbus.in/bus-tickets/solapur-to-goa,Sharma Travels,Bharat Benz A/C Sleeper (2+1),23:00,10h 00m,09:00,4.6,INR 1300,NaN
413,Solapur to Goa,https://www.redbus.in/bus-tickets/solapur-to-goa,Sharma Travels,AC Sleeper (2+1),22:00,10h 00m,08:00,4.4,INR 1200,NaN
414,Solapur to Goa,https://www.redbus.in/bus-tickets/solapur-to-goa,Sharma Travels,AC Sleeper (2+1),22:30,10h 00m,08:30,4.0,INR 1200,NaN


In [6]:
df.columns

Index(['route', 'link', 'Bus Name', 'Bus Type', 'Departing Time',
       'Duration Time', 'Reaching Time', 'Star Rating', 'Price',
       'Seats Availability'],
      dtype='object')